In [3]:
import pandas as pd
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

/Users/reo/Documents/Reo/data-science/projects/barista-bench/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from barista_bench import config

import os

In [13]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Tiny, fast, smart enough for JSON
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

In [3]:
DATA_DIR = "../data"  


test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

with open(f"{DATA_DIR}/menu.md", "r") as f:
    MENU_TEXT = f.read()

# --- 3. PREPARE FEW-SHOT EXAMPLES ---
# We pick 3 examples from the training set to teach the model the pattern.
# We choose a mix of simple and complex rows if possible, but random 3 works well.
examples = train_df.sample(3, random_state=42)  # Fixed seed for consistency

FEW_SHOT_CONTEXT = "Here are 3 examples of how to correctly parse orders:\n\n"
for _, row in examples.iterrows():
    FEW_SHOT_CONTEXT += f"Customer: \"{row['order']}\"\n"
    FEW_SHOT_CONTEXT += f"Response: {row['expected_json']}\n\n"

print("--- Generated Few-Shot Context ---")
print(FEW_SHOT_CONTEXT)
print("----------------------------------")

# --- 4. LOAD MODEL ---
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,  # Half-precision for 2x speed
    device_map="auto",  # Auto-assign to GPU
)


# --- 5. THE SOLVER FUNCTION ---
def solve_with_few_shot(order_text):
    # Construct the Prompt
    # We combine: Role + Menu + Examples + New Question
    messages = [
        {
            "role": "system",
            "content": f"""You are a Barista POS system.
Your job is to parse natural language orders into valid JSON.

MENU & RULES:
{MENU_TEXT}

FORMAT:
Return ONLY a JSON object. Do not explain.
Schema: {{ "drink": "Name", "size": "Size", "final_milk": "Milk/None", "modifiers": ["List"], "food_items": ["List"], "total_price": 0.00 }}
""",
        },
        # We inject the few-shot examples as the first user message
        {
            "role": "user",
            "content": f'{FEW_SHOT_CONTEXT}\n\nNow, solve this new order:\nCustomer: "{order_text}"\nResponse:',
        },
    ]

    # Tokenize
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate
    # We use a low temperature (0.1) to make the model factual and mathematical
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,  # Limit output length to prevent rambling
        temperature=0.1,
        do_sample=True,
    )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract JSON part (Basic cleanup)
    if "assistant" in response:
        response = response.split("assistant")[-1]

    try:
        # Locate the JSON brackets
        start = response.find("{")
        end = response.rfind("}") + 1
        if start != -1 and end != -1:
            return response[start:end]
        else:
            return "{}"  # Return empty JSON on failure
    except:
        return "{}"


# --- 6. MAIN EXECUTION LOOP ---
results = []
print(f"Processing {len(test_df)} orders...")

# Tqdm adds a progress bar so you know how long it will take
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    json_out = solve_with_few_shot(row["order"])
    results.append({"id": row["id"], "predicted_json": json_out})

# --- 7. SAVE SUBMISSION ---
submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)
print("✅ Success! submission.csv created.")

NameError: name 'pd' is not defined

In [7]:
! ls ../

notebooks


In [10]:
DEVICE

'cpu'

In [12]:
torch.backends.mps.is_available()

True